## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
)


# 에이전트 디버깅(Agent Debugging)

agent 시스템이 이상한 답을 냈을 때, "모델이 멍청하다"고 끝내면 아무것도 고칠 수 없다. 이 노트북은 trace, node input/output, failure table을 이용해 어디서 문제가 시작됐는지 단계별로 추적하는 디버깅 습관을 만든다.

## 학습 목표
- trace를 표로 바꿔 읽는 법을 이해한다.
- node별 input/output을 따로 보는 이유를 설명할 수 있다.
- latency를 병목 신호로 해석하되, 숫자를 맥락과 함께 읽는 법을 안다.
- failure table과 trace를 연결해 실제 디버깅 순서를 말할 수 있다.


## 개념 설명

디버깅의 기본 원칙은 최종 답변에서 거꾸로 거슬러 올라가는 것이다. 답이 이상하면 verifier를 보고, verifier 입력이 이상하면 synthesis를 보고, synthesis 입력이 이상하면 retrieval을 보는 식이다. 이 저장소는 이런 역추적을 쉽게 하려고 trace와 state validation을 명시적으로 남긴다.

**목적**
- 디버깅을 감이 아니라 관찰 가능한 데이터 흐름으로 접근한다.

**핵심 로직**
- trace는 전체 경로를 보여준다.
- node input/output view는 특정 단계만 확대해 준다.
- failure table은 반복 패턴을 찾게 해 준다.

**결과 해석 가이드**
- 이 notebook의 목표는 "한 번의 실패 설명"보다 "다음에도 같은 방식으로 찾는 절차"를 익히는 것이다.

**💡 면접 포인트**
- "Agent debugging은 prompt 감상보다 trace inspection에 가깝다"고 말할 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

아래 셀은 retriever, run_workflow, trace display helper를 준비한다. 여기서 중요한 점은 trace를 그냥 JSON으로 보지 않고, DataFrame으로 바꿔 단계별로 읽을 수 있게 한 점이다. 도구가 단순할수록 디버깅 습관이 잘 자리 잡는다.

**목적**
- debugging 실습에 필요한 helper를 세팅한다.

**핵심 로직**
- `display_trace`, `display_node_inputs`, `display_node_outputs`가 서로 다른 확대 수준의 뷰를 제공한다.

**실제 소스 코드: validate_state() — src/state_validation.py**
```python
NODE_REQUIREMENTS: dict[str, dict[str, type[Any]]] = {
    "normalize_query": {"normalized_query": str},
    "classify_query": {"query_type": str, "requires_tools": bool},
    "make_plan": {"plan": list},
    "retrieve_docs": {"retrieved_docs": list},
    "retrieve_memories": {"retrieved_memories": list},
    "decide_tools": {"tool_requests": list},
    "run_tools": {"tool_outputs": list},
    "synthesize_answer": {"draft_answer": str, "citations": list},
    "verify_grounding": {"verification_result": object},
    "fallback_or_finalize": {"final_answer": str, "final_status": str},
    "update_memory": {"memory_updates": list},
}


def _is_invalid(value: Any, expected_type: type[Any]) -> bool:
    if value is None:
        return True
    if expected_type is str:
        return not isinstance(value, str) or not value.strip()
    if expected_type is bool:
        return not isinstance(value, bool)
    if expected_type is list:
        return not isinstance(value, list)
    return False


def validate_state(state: AgentState) -> None:
    completed_nodes = {str(entry.get("node", "")) for entry in state.get("trace", [])}
    issues: list[str] = []

    for node_name, requirements in NODE_REQUIREMENTS.items():
        if node_name not in completed_nodes:
            continue
        for key, expected_type in requirements.items():
            if key not in state:
                issues.append(f"{key} missing after {node_name}")
                continue
            if _is_invalid(state[key], expected_type):
                issues.append(f"{key} invalid after {node_name}")

    if issues:
        issue_text = "; ".join(issues)
        raise StateValidationError(f"State validation failed: {issue_text}")
```

**코드 읽기 포인트**
- state validation은 "조용히 잘못된 상태"를 남기지 않기 위한 안전장치다.
- `NODE_REQUIREMENTS`가 각 node 완료 후 필요한 state key를 정의한다.
- 디버깅은 trace를 보기 전에 state contract가 깨졌는지부터 보는 습관이 중요하다.

**결과 해석 가이드**
- 준비 셀 자체는 조용하지만, 이후 어떤 뷰를 쓰는지 이해하는 것이 핵심이다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.evaluator import extract_failure_cases, run_evaluation_suite
from src.ingestion import build_demo_index
from src.trace_debug import display_node_inputs, display_node_outputs, display_trace
from src.workflow import run_workflow

pd.set_option('display.max_colwidth', 140)
retriever = build_demo_index(persist=False)


## 디버깅 시나리오 시작

먼저 정상적인 happy path와 abstain path를 각각 실행해 본다. 디버깅은 실패만 보는 것이 아니라, 정상 경로와 비교하는 기준선을 만드는 일도 포함한다. 그래야 어떤 상태가 "이상한 것"인지 판단할 수 있다.

**목적**
- 비교 기준이 되는 두 실행 결과를 만든다.

**핵심 로직**
- 하나는 계산이 필요한 질문, 하나는 문서 밖 질문으로 구성해 서로 다른 최종 상태를 본다.

**결과 해석 가이드**
- `happy_status=answered`, `abstain_status=abstained`가 나오면 두 경로가 제대로 분기한 것이다.


In [ ]:
happy_state = run_workflow('How many days are in the pilot window?', retriever=retriever)
abstain_state = run_workflow('Who is the current CEO of the company?', retriever=retriever)
{
    'happy_status': happy_state['final_status'],
    'abstain_status': abstain_state['final_status'],
    'happy_steps': len(happy_state['trace']),
    'abstain_steps': len(abstain_state['trace']),
}


## 전체 trace 읽기

trace JSON을 바로 읽으면 눈으로 따라가기 어렵다. `trace_to_debug_frame()`는 이를 표로 바꿔 node, latency, inputs, outputs, timestamp를 한 줄씩 보여 준다. 이 변환 단계가 있어야 사람 중심 디버깅이 가능해진다.

**목적**
- trace를 사람이 읽기 쉬운 디버그 테이블로 바꾼다.

**핵심 로직**
- 각 trace entry를 순회하며 step 번호를 붙인다.
- timestamp가 있으면 이전 step과의 시간 차를 latency fallback으로 계산할 수도 있다.
- `inputs`, `outputs`는 긴 JSON을 잘라 문자열로 직렬화한다.

**실제 소스 코드: trace_to_debug_frame() — src/trace_debug.py**
```python
def trace_to_debug_frame(trace: list[dict[str, Any]]) -> pd.DataFrame:
    previous_timestamp: datetime | None = None
    rows: list[dict[str, Any]] = []
    for index, entry in enumerate(trace, start=1):
        current_timestamp = _parse_timestamp(str(entry.get("timestamp", "")))
        latency = entry.get("latency")
        if not isinstance(latency, (int, float)) and previous_timestamp is not None and current_timestamp is not None:
            latency = round((current_timestamp - previous_timestamp).total_seconds(), 6)
        previous_timestamp = current_timestamp or previous_timestamp
        rows.append(
            {
                "step": index,
                "node": str(entry.get("node", "")),
                "latency": round(float(latency), 6) if isinstance(latency, (int, float)) else None,
                "inputs": _serialize(entry.get("inputs", {})),
                "outputs": _serialize(entry.get("outputs", entry.get("payload", {}))),
                "timestamp": str(entry.get("timestamp", "")),
            }
        )
    return pd.DataFrame(rows, columns=["step", "node", "latency", "inputs", "outputs", "timestamp"])
```

**실제 소스 코드: display_trace() — src/trace_debug.py**
```python
def display_trace(trace: list[dict[str, Any]], render: bool = True) -> pd.DataFrame:
    frame = trace_to_debug_frame(trace)
    if render:
        try:  # pragma: no branch - notebook convenience
            from IPython.display import display

            display(frame)
        except Exception:  # pragma: no cover
            print(frame.to_string(index=False))
    return frame
```

**코드 읽기 포인트**
- `_serialize()`가 긴 JSON을 잘라줘 notebook 테이블이 읽기 쉬워진다.
- `latency`가 없을 때 timestamp 차이로 보정하는 로직이 있어 구버전 trace도 어느 정도 읽을 수 있다.
- `display_trace()`는 render 여부를 받아 notebook display와 programmatic use를 둘 다 지원한다.

**결과 해석 가이드**
- 표에서 step 순서를 따라가며 최초로 이상한 output이 나온 node를 찾는 것이 기본 디버깅 루틴이다.
- `inputs`와 `outputs`를 함께 보지 않으면 노드가 틀렸는지, upstream이 이미 틀렸는지 구분하기 어렵다.


In [ ]:
debug_trace_frame = display_trace(happy_state['trace'], render=False)
debug_trace_frame


## Trace 성능 분석(Trace Performance Analysis)

latency는 단순한 속도 숫자가 아니라, 어느 node가 시스템 비용을 주도하는지 알려 주는 힌트다. 다만 숫자만 보면 오해할 수 있다. retrieval이 normalize보다 느린 것은 자연스럽지만, 작은 tool call이 지나치게 느리다면 병목 신호일 수 있다.

**목적**
- node별 latency를 병목 관점에서 읽는다.

**핵심 로직**
- 같은 trace에서 latency 컬럼만 뽑아 비교한다.

**실제 소스 코드: display_trace() — src/trace_debug.py**
```python
def display_trace(trace: list[dict[str, Any]], render: bool = True) -> pd.DataFrame:
    frame = trace_to_debug_frame(trace)
    if render:
        try:  # pragma: no branch - notebook convenience
            from IPython.display import display

            display(frame)
        except Exception:  # pragma: no cover
            print(frame.to_string(index=False))
    return frame
```

**코드 읽기 포인트**
- latency가 높다고 자동으로 나쁜 것은 아니다. 질문은 "그 node의 역할 대비 과도한가"이다.
- 비슷한 질문에서 특정 node만 유독 흔들리면 비결정성이나 외부 의존성 문제를 의심할 수 있다.

**결과 해석 가이드**
- ms 또는 초 단위 절대값보다, 다른 node 대비 상대적 크기를 먼저 본다.
- 반복 실행에서 같은 node latency가 들쭉날쭉하면 그 node를 우선 계측해야 한다.


In [ ]:
performance_frame = display_trace(happy_state['trace'], render=False)[['node', 'latency', 'inputs', 'outputs']]
performance_frame

latency가 높다고 해서 자동으로 나쁜 것은 아니다. 중요한 질문은 그 시간이 해당 작업에 비해 자연스러운지 여부다. 예를 들어 retrieval이 normalization보다 느린 것은 자연스럽지만, tool이 거의 아무 일도 하지 않는데 유난히 느리다면 병목 후보가 된다. 따라서 성능 분석은 절대 숫자보다 **상대 비교**와 **반복 패턴**으로 읽는 편이 안전하다.


## 노드 입력 점검(Node Inputs)

어떤 node가 잘못된 답을 냈다고 보여도, 실제 원인은 그 node 입력이 이미 오염돼 있었기 때문일 수 있다. 그래서 디버깅에서는 output만 보지 않고 input을 따로 보는 뷰가 꼭 필요하다. `retrieve_docs`에 엉뚱한 query가 들어갔다면 retriever가 아니라 upstream normalization/classification이 문제다.

**목적**
- 특정 node가 받은 입력만 떼어 본다.

**핵심 로직**
- trace에서 node 이름이 일치하는 entry만 골라 `inputs` 컬럼으로 보여준다.

**실제 소스 코드: display_node_inputs() — src/trace_debug.py**
```python
def display_node_inputs(trace: list[dict[str, Any]], node_name: str, render: bool = True) -> pd.DataFrame:
    rows = [
        {"step": index, "node": entry.get("node", ""), "inputs": _serialize(entry.get("inputs", {}))}
        for index, entry in enumerate(trace, start=1)
        if entry.get("node") == node_name
    ]
    frame = pd.DataFrame(rows, columns=["step", "node", "inputs"])
    if render:
        try:  # pragma: no branch
            from IPython.display import display

            display(frame)
        except Exception:  # pragma: no cover
            print(frame.to_string(index=False))
    return frame
```

**코드 읽기 포인트**
- 특정 node를 반복 실행한 trace가 있다면 step이 여러 줄로 나올 수 있다.
- 입력 점검은 "이 node가 잘못했는가"보다 "이 node가 잘못된 입력을 받았는가"를 묻는 단계다.

**결과 해석 가이드**
- `normalized_query`, `top_k` 같은 입력이 기대와 다르면 그 앞 단계부터 다시 봐야 한다.


In [ ]:
display_node_inputs(happy_state['trace'], 'retrieve_docs', render=False)


## 노드 출력 점검(Node Outputs)

입력이 정상이라면 다음은 출력을 본다. 예를 들어 `retrieve_docs`의 output에서 `results=0`이거나 source가 엉뚱하면 retrieval 자체가 문제다. 반대로 retrieval output은 멀쩡한데 final answer가 이상하면 synthesis 이후를 보면 된다.

**목적**
- 특정 node가 실제로 무엇을 만들었는지 본다.

**핵심 로직**
- trace에서 node별 `outputs`만 추출해 표로 만든다.

**실제 소스 코드: display_node_outputs() — src/trace_debug.py**
```python
def display_node_outputs(trace: list[dict[str, Any]], node_name: str, render: bool = True) -> pd.DataFrame:
    rows = [
        {"step": index, "node": entry.get("node", ""), "outputs": _serialize(entry.get("outputs", {}))}
        for index, entry in enumerate(trace, start=1)
        if entry.get("node") == node_name
    ]
    frame = pd.DataFrame(rows, columns=["step", "node", "outputs"])
    if render:
        try:  # pragma: no branch
            from IPython.display import display

            display(frame)
        except Exception:  # pragma: no cover
            print(frame.to_string(index=False))
    return frame
```

**코드 읽기 포인트**
- input/output 뷰를 분리하면 원인과 결과를 섞지 않고 볼 수 있다.
- 특히 retrieval, verify, fallback 노드는 output 해석이 곧 다음 action 결정으로 이어진다.

**결과 해석 가이드**
- `retrieve_docs` output의 source 목록은 retrieval 품질 판단의 핵심이다.
- `verify_grounding` output의 unsupported claim은 hallucination 디버깅의 핵심 단서다.


In [ ]:
display_node_outputs(happy_state['trace'], 'retrieve_docs', render=False)


## 실험

이제 happy path와 abstain path를 같은 표에서 비교한다. 이 실험은 "좋은 실행"과 "안전한 거절"을 모두 정상 동작으로 보는 관점을 만든다. 디버깅은 answered만 정답으로 두지 않아야 한다.

**목적**
- 서로 다른 최종 상태를 비교해 verifier와 fallback의 역할을 읽는다.

**결과 해석 가이드**
- coverage_score와 unsupported_claims 개수를 함께 보면, 왜 answered 혹은 abstained가 됐는지 더 잘 이해할 수 있다.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            'query': happy_state['user_query'],
            'final_status': happy_state['final_status'],
            'coverage_score': happy_state['verification_result'].coverage_score,
            'unsupported_claims': len(happy_state['verification_result'].unsupported_claims),
        },
        {
            'query': abstain_state['user_query'],
            'final_status': abstain_state['final_status'],
            'coverage_score': abstain_state['verification_result'].coverage_score,
            'unsupported_claims': len(abstain_state['verification_result'].unsupported_claims),
        },
    ]
)
comparison


디버깅은 반복 evaluation과 함께 볼 때 더 강해진다. 한두 개 trace만 보면 우연인지 패턴인지 알기 어렵기 때문이다. 이 셀은 작은 evaluation run을 돌린 뒤 failure row만 추려서, **어떤 질문이 반복적으로 문제를 일으키는지** 보게 해 준다. 즉 single-case debugging에서 multi-run debugging으로 넘어가는 단계라고 볼 수 있다.


In [ ]:
results, summary = run_evaluation_suite(repeats=1, persist_outputs=True)
failures = extract_failure_cases(results)
failures[['system', 'question_id', 'question', 'failure_type', 'predicted_status']].head(10)


## 결과 해석

마지막 분석 표는 어떤 디버깅 뷰를 언제 써야 하는지 정리한 것이다. 실제 현장에서는 보통 `failure_table -> full_trace -> node_inputs -> node_outputs` 순으로 좁혀 가는 경우가 많다. 먼저 반복 패턴을 찾고, 그다음 개별 케이스를 깊게 파는 방식이다.

**목적**
- 디버깅 루틴을 재사용 가능한 절차로 정리한다.

**결과 해석 가이드**
- `full_trace`는 전체 경로 파악, `node_inputs`는 upstream 오염 확인, `node_outputs`는 해당 node 자체의 결과 확인, `failure_table`은 반복 패턴 찾기에 적합하다.

**💡 면접 포인트**
- "답변이 이상할 때는 final answer부터 보지 않고, 가장 먼저 이상해진 trace node를 찾는다"고 설명하면 좋다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'debugging_view': 'full_trace', 'use_case': 'understand the whole execution path'},
        {'debugging_view': 'node_inputs', 'use_case': 'inspect what information reached a node'},
        {'debugging_view': 'node_outputs', 'use_case': 'inspect what the node actually produced'},
        {'debugging_view': 'failure_table', 'use_case': 'spot recurring issues across many runs'},
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 agent debugging은 감각적 추측이 아니라, trace와 state contract를 이용한 구조적 분석이라는 점을 확인했다. `trace_to_debug_frame()`와 `display_trace()`는 전체 실행 경로를 보여주고, `display_node_inputs()`와 `display_node_outputs()`는 특정 node를 확대해서 원인을 좁히게 해 준다. 여기에 failure table을 결합하면 단일 케이스 분석과 반복 패턴 분석이 연결된다.

결국 중요한 것은 "어디서 처음 이상해졌는가"를 찾는 습관이다. retrieval이 틀렸는지, synthesis가 과장했는지, verifier가 너무 엄격한지, fallback이 너무 느슨한지를 node 단위로 나눠 봐야 빠르게 고칠 수 있다.

**💡 면접 포인트**
- "Agent debugging은 trace-first 접근이 효과적이다. 최종 답변보다 최초 이상 node를 찾는 것이 중요하다."
- "Latency도 품질과 함께 봐야 한다. 느린 node가 항상 나쁜 것은 아니고, 역할 대비 비정상적으로 느릴 때가 병목이다."
- "Failure table과 trace를 함께 보면 개별 오류와 반복 패턴을 동시에 잡을 수 있다."
